
python -m venv venv

venv\Scripts\activate


pip install --upgrade pip

pip install requests beautifulsoup4 playwright deep-translator pyttsx3 gTTS pydub

playwright install
<div style="direction: rtl; text-align: justify; font-family: 'Vazirmatn', Tahoma, sans-serif; font-size: 15px; line-height: 1.8;">
. نصب ffmpeg برای پردازش صدا

کتابخانه pydub برای ترکیب فایل‌های صوتی نیازمند ffmpeg است.
در ویندوز:

    دانلود ffmpeg از سایت رسمی: https://ffmpeg.org/download.html

    فایل را استخراج کرده و مسیر پوشه bin را به Path سیستم اضافه کنید.

ffmpeg -version


<div style="direction: rtl; text-align: justify; font-family: 'Vazirmatn', Tahoma, sans-serif; font-size: 15px; line-height: 1.8;">
مرحله ۱ — دریافت و ذخیره‌سازی لینک‌های بخش دوز مصرفی داروها از Drugs.com

این کد وظیفه دارد که از صفحه اصلی بخش Dosage سایت drugs.com تمام لینک‌های مربوط به صفحات توضیحات دوز مصرفی داروها را جمع‌آوری کرده و آن‌ها را در یک فایل متنی ذخیره کند.
کتابخانه‌های استفاده‌شده

    os → برای ساخت مسیرها و مدیریت فایل‌ها/پوشه‌ها

    requests → برای ارسال درخواست HTTP و دریافت محتوای صفحات وب

    BeautifulSoup (از bs4) → برای تجزیه HTML و استخراج لینک‌ها

تعریف ثابت‌ها

    BASE_URL → آدرس اصلی سایت

    START_URL → آدرس صفحه شروع (لیست دوز داروها)

    SAVE_DIR → مسیر پوشه ذخیره‌سازی لینک‌ها

    SAVE_PATH → مسیر کامل فایل متنی که لینک‌ها داخل آن ذخیره می‌شوند

تابع scrape_dosage_links()

    اتصال به صفحه شروع

        با requests.get به آدرس START_URL درخواست ارسال می‌شود.

        اگر خطا یا کد وضعیت غیر از 200 دریافت شود، اجرای برنامه متوقف می‌شود.

    تجزیه HTML با BeautifulSoup

        محتوای صفحه با BeautifulSoup به فرمت HTML-Parser خوانده می‌شود.

        همه تگ‌های <a> که ویژگی href دارند استخراج می‌شود.

    فیلتر کردن لینک‌های موردنظر

        فقط لینک‌هایی که با /dosage/ شروع و با .html تمام می‌شوند انتخاب می‌شوند.

        این لینک‌ها به آدرس کامل (BASE_URL + لینک نسبی) تبدیل و به لیست اضافه می‌شوند.

    ذخیره‌سازی لینک‌ها در فایل

        اگر هیچ لینکی پیدا نشود، پیغام هشدار داده می‌شود.

        در غیر این صورت، پوشه SAVE_DIR ساخته می‌شود (اگر وجود نداشته باشد).

        تمام لینک‌ها در فایل متنی dosage_links.txt ذخیره می‌شوند.

    بررسی فایل ذخیره‌شده

        پس از ذخیره‌سازی، فایل دوباره خوانده می‌شود.

        تعداد کل لینک‌ها و نمونه‌ای از ۳ لینک اول چاپ می‌شود تا صحت عملیات تأیید شود.

In [ ]:
import os
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.drugs.com"
START_URL = BASE_URL + "/dosage/"

SAVE_DIR = os.path.join("MegaMedSpeechDataset", "1_scraped_english_data")
SAVE_PATH = os.path.join(SAVE_DIR, "dosage_links.txt")

def scrape_dosage_links():
    print("[*] Connecting to dosage index page...")

    try:
        response = requests.get(START_URL, timeout=10)
    except Exception as e:
        print(f"[!] Error fetching the page: {e}")
        return

    if response.status_code != 200:
        print(f"[!] Unexpected status code: {response.status_code}")
        return

    soup = BeautifulSoup(response.text, "html.parser")

    all_links = soup.find_all("a", href=True)
    dosage_links = []

    for a in all_links:
        href = a['href']
        if href.startswith("/dosage/") and href.endswith(".html"):
            full_url = BASE_URL + href.strip()
            dosage_links.append(full_url)

    if not dosage_links:
        print("[!] No dosage links found. Page may have no visible links.")
        return

    print(f"[✓] Found {len(dosage_links)} dosage links. Attempting to save...")

    os.makedirs(SAVE_DIR, exist_ok=True)

    try:
        with open(SAVE_PATH, "w", encoding="utf-8") as f:
            for link in dosage_links:
                f.write(link + "\n")
        print(f"[✓] Successfully saved to: {os.path.abspath(SAVE_PATH)}")
    except Exception as e:
        print(f"[!] Error writing file: {e}")

    try:
        with open(SAVE_PATH, "r", encoding="utf-8") as f:
            lines = f.readlines()
            print(f"[✓] File contains {len(lines)} lines.")
            if len(lines) > 0:
                print("[Sample lines]:")
                for line in lines[:3]:
                    print(line.strip())
            else:
                print("[!] File is empty after write!")
    except Exception as e:
        print(f"[!] Error reading file after write: {e}")

if __name__ == "__main__":
    scrape_dosage_links()

<div style="direction: rtl; text-align: justify; font-family: 'Vazirmatn', Tahoma, sans-serif; font-size: 15px; line-height: 1.8;">
مرحله ۲ — خزش (Crawling) صفحات و استخراج متن دوز مصرفی داروها

این کد از لینک‌های جمع‌آوری‌شده در مرحله ۱ استفاده می‌کند، به هر صفحه می‌رود و محتوای اصلی مربوط به دوز مصرفی را استخراج کرده و در یک فایل JSON ذخیره می‌کند.
کتابخانه‌های استفاده‌شده

    json → برای ذخیره داده‌ها در فرمت JSON

    os, Pathlib → برای مدیریت مسیرها و فایل‌ها

    time → برای ایجاد تأخیر بین درخواست‌ها (کاهش فشار روی سرور)

    BeautifulSoup → برای تجزیه HTML و استخراج محتوای متنی

    playwright → برای بارگذاری کامل صفحات وب (شامل محتوای تولیدشده با جاوااسکریپت)

مسیرهای ورودی و خروجی

    input_file → مسیر فایل متنی که لیست لینک‌ها از مرحله قبل در آن ذخیره شده است.

    output_dir → پوشه ذخیره‌سازی خروجی.

    output_path → مسیر فایل JSON نهایی که متن‌های دوز مصرفی را نگه می‌دارد.

اگر پوشه خروجی وجود نداشته باشد، با os.makedirs(..., exist_ok=True) ساخته می‌شود.


In [ ]:
import json
import os
import time
from pathlib import Path
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright

input_file = "1_scraped_english_data/dosage_links.txt"
output_dir = "1_scraped_english_data"
output_path = os.path.join(output_dir, "dosages.json")

os.makedirs(output_dir, exist_ok=True)

with open(input_file, "r") as f:
    links = [line.strip() for line in f if line.strip()]

data = []

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()

    for i, url in enumerate(links, 1):
        print(f"→ ({i}/{len(links)}) Crawling: {url}")
        try:
            page.goto(url, timeout=60000)
            page.wait_for_load_state("networkidle")
            html = page.content()
            soup = BeautifulSoup(html, "html.parser")

            main_content = soup.find("div", class_="contentBox") or soup.find("div", id="content")
            if main_content:
                text = main_content.get_text(separator="\n", strip=True)
                data.append({"url": url, "dosage_text": text})
                print(f"   [✓] Saved.")
            else:
                print(f"   [!] No content found.")
                data.append({"url": url, "dosage_text": None})
        except Exception as e:
            print(f"   [!] Failed: {e}")
            data.append({"url": url, "dosage_text": None})
        
        time.sleep(2)  

    browser.close()

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"\n[✓] Done! Saved to: {output_path}")

<div style="direction: rtl; text-align: justify; font-family: 'Vazirmatn', Tahoma, sans-serif; font-size: 15px; line-height: 1.8;">
مرحله ۳ — ترجمه متون دوز مصرفی به زبان فارسی

در این مرحله، متن‌های انگلیسی جمع‌آوری‌شده از مرحله دوم با استفاده از سرویس Google Translate به زبان فارسی ترجمه و در همان فایل JSON ذخیره می‌شوند.
کتابخانه‌های استفاده‌شده

    deep_translator.GoogleTranslator → برای ترجمه متن‌ها با استفاده از Google Translate API غیررسمی

    json → برای خواندن و ذخیره داده‌ها در قالب JSON

    re → برای تقسیم متن به بخش‌های کوچک‌تر با استفاده از عبارات منظم (Regular Expressions)

مسیر ورودی/خروجی

    input_output_file → مسیر فایل JSON که در مرحله دوم ساخته شده (dosages.json).
    این فایل هم برای خواندن متن‌ها و هم برای نوشتن ترجمه‌ها استفاده می‌شود.


    تقسیم متن به بخش‌های کوچک‌تر

به دلیل محدودیت طول ورودی سرویس ترجمه، متن‌های بلند باید شکسته شوند.


            '''def split_text(text, max_chars=450):
                parts = re.split(r'(?<=[.!?])\s+|\n+', text)
                ...'''

    متن بر اساس نقطه، علامت سوال، علامت تعجب یا خط جدید شکسته می‌شود.

    هر بخش تا حداکثر ۴۵۰ کاراکتر جمع‌آوری می‌شود تا از خطاهای ناشی از متن بیش از حد طولانی جلوگیری شود.

    خروجی این تابع یک لیست از تکه‌های متن است.

In [ ]:
from deep_translator import GoogleTranslator
import json
import re

input_output_file = "1_scraped_english_data/dosages.json"

with open(input_output_file, "r", encoding="utf-8") as f:
    data = json.load(f)

translator = GoogleTranslator(source="en", target="fa")

def split_text(text, max_chars=450):
    parts = re.split(r'(?<=[.!?])\s+|\n+', text)
    chunks = []
    current = ""

    for part in parts:
        if len(current) + len(part) < max_chars:
            current += part + " "
        else:
            chunks.append(current.strip())
            current = part + " "

    if current.strip():
        chunks.append(current.strip())
    
    return chunks

for i, item in enumerate(data):
    url = item.get("url", "N/A")
    text = item.get("dosage_text", "")

    if not isinstance(text, str) or len(text.strip()) < 10:
        print(f"[{i+1}] ⏩ Skipped: Invalid text.")
        item["translated_text"] = ""
        continue

    try:
        chunks = split_text(text)
        translated_chunks = []

        for j, chunk in enumerate(chunks):
            translated_chunk = translator.translate(chunk)
            translated_chunks.append(translated_chunk)
        
        final_translation = "\n".join(translated_chunks).strip()
        item["translated_text"] = final_translation

        print(f"[{i+1}/{len(data)}]  Translated [{len(chunks)} chunks]")

    except Exception as e:
        print(f"[{i+1}]  Error: {e}")
        item["translated_text"] = ""

# ذخیره فایل با ترجمه‌ها
with open(input_output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("\n ترجمه کامل شد. فایل با موفقیت ذخیره شد.")



<div style="direction: rtl; text-align: justify; font-family: 'Vazirmatn', Tahoma, sans-serif; font-size: 15px; line-height: 1.8;">

 مرحله چهارم
 مراحل اصلی

    تعریف حداکثر طول متن (MAX_CHARS = 500)

        چون موتورهای TTS (تبدیل متن به گفتار) معمولاً محدودیت طول ورودی دارند، متن‌ها را به بخش‌های کوچک‌تر می‌شکنیم.

    تابع split_text()

        متن فارسی یا انگلیسی را بر اساس پایان جمله (نقطه، علامت سوال، علامت تعجب و «؟») تقسیم می‌کند.

        بخش‌ها را طوری گروه‌بندی می‌کند که هر قطعه حداکثر ۵۰۰ کاراکتر باشد.

    تولید صدای فارسی (save_tts_fa)

        از کتابخانه pyttsx3 برای TTS فارسی استفاده می‌شود (نیازمند نصب ویس فارسی روی سیستم).

        اگر صدای فارسی پیدا نشود، پیغام خطا داده و رد می‌شود.

        هر قطعه متن فارسی جداگانه به MP3 ذخیره می‌شود.

        سپس تمام قطعات با pydub به یک فایل MP3 نهایی ترکیب می‌شوند.

        فایل‌های موقت پاک می‌شوند.

    تولید صدای انگلیسی (save_tts_en)

        از gTTS (Google Text-to-Speech) استفاده می‌شود.

        متن کامل انگلیسی یک‌جا به MP3 تبدیل می‌شود (چون gTTS به صورت آنلاین و دقیق‌تر کار می‌کند).

    پردازش همه داده‌ها

        کد فایل dosages.json را می‌خواند.

        برای هر دارو، نام فایل صوتی بر اساس نام موجود در URL ساخته می‌شود.

        اگر متن فارسی طولانی‌تر از ۵۰ کاراکتر باشد → به صوت تبدیل می‌شود.

        اگر متن انگلیسی طولانی‌تر از ۵۰ کاراکتر باشد → به صوت تبدیل می‌شود.

        در غیر این صورت، پیام "Skipped" چاپ می‌شود.

نتیجه

در پایان اجرای این مرحله:

    در پوشه fa برای هر دارو یک فایل صوتی MP3 حاوی نسخه فارسی متن وجود دارد.

    در پوشه en برای هر دارو یک فایل صوتی MP3 حاوی نسخه انگلیسی متن وجود دارد.

این بخش آخرین مرحله از چرخه خزش → استخراج → ترجمه → تبدیل به صوت است و دیتاست نهایی آماده استفاده در پروژه‌های پردازش گفتار یا مدل‌های هوش مصنوعی می‌شود.

این قسمت ار استپ شد ومنتظر اراِه یک مدل تبدیل متن به ویس فارسی از دوستان هستیم

In [ ]:
import os
import json
import re
import pyttsx3
from gtts import gTTS
from pathlib import Path
from pydub import AudioSegment

MAX_CHARS = 500
input_file = "1_scraped_english_data/dosages.json"
output_dir_fa = "2_generated_audio/fa"
output_dir_en = "2_generated_audio/en"
os.makedirs(output_dir_fa, exist_ok=True)
os.makedirs(output_dir_en, exist_ok=True)

def split_text(text):
    sentences = re.split(r'(?<=[.!؟])\s+', text)
    chunks = []
    current = ""
    for sentence in sentences:
        if len(current) + len(sentence) <= MAX_CHARS:
            current += " " + sentence
        else:
            if current.strip():
                chunks.append(current.strip())
            current = sentence
    if current.strip():
        chunks.append(current.strip())
    return chunks

def save_tts_fa(chunks, output_path):
    try:
        engine = pyttsx3.init()
        
        voices = engine.getProperty('voices')
        fa_voice = None
        for voice in voices:
            if 'fa' in voice.id or 'iran' in voice.name.lower() or 'persian' in voice.name.lower():
                fa_voice = voice.id
                break
        
        if not fa_voice:
            print(" Persian is not supported. Please install Persian Voice.")
            return False

        engine.setProperty('voice', fa_voice)
        temp_files = []

        for i, chunk in enumerate(chunks):
            temp_path = f"{output_path}_temp_{i}.mp3"
            engine.save_to_file(chunk, temp_path)
            temp_files.append(temp_path)
        engine.runAndWait()

        combined = AudioSegment.empty()
        for tf in temp_files:
            combined += AudioSegment.from_file(tf)
        combined.export(output_path, format="mp3")

        for tf in temp_files:
            os.remove(tf)

        return True
    except Exception as e:
        print(f" FA combine failed: {e}")
        return False

def save_tts_en(text, output_path):
    try:
        tts = gTTS(text, lang="en")
        tts.save(output_path)
        return True
    except Exception as e:
        print(f" EN failed: {e}")
        return False

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

for i, entry in enumerate(data):
    url = entry.get("url", "")
    name = url.strip("/").split("/")[-1].replace(".html", "")

    text_fa = entry.get("translated_text", "")
    text_en = entry.get("dosage_text", "")

    if isinstance(text_fa, str) and len(text_fa.strip()) > 50:
        chunks = split_text(text_fa)
        fa_path = os.path.join(output_dir_fa, f"{name}.mp3")
        if save_tts_fa(chunks, fa_path):
            print(f"[{i+1}]  FA saved: {name}.mp3")
        else:
            print(f"[{i+1}]  FA failed: {name}")
    else:
        print(f"[{i+1}]  Skipped FA (too short): {name}")

    if isinstance(text_en, str) and len(text_en.strip()) > 50:
        en_path = os.path.join(output_dir_en, f"{name}.mp3")
        if save_tts_en(text_en, en_path):
            print(f"[{i+1}]  EN saved: {name}.mp3")
        else:
            print(f"[{i+1}]  EN failed: {name}")
    else:
        print(f"[{i+1}] Skipped EN (too short): {name}")
